# 类继承关系
```mermaid
classDiagram
    class MetaStatsBuilderMixin {
        <<metaclass>>
    }
    class MetaPlotsBuilderMixin {
        <<metaclass>>
    }
    class StatsBuilderMixin
    class MetaMappedArray
    class PlotsBuilderMixin

    class Wrapping
    class MappedArray

    
    %% 继承关系
    MetaStatsBuilderMixin <|-- StatsBuilderMixin : metaclass
    MetaStatsBuilderMixin <|-- MetaMappedArray : metaclass
    MetaPlotsBuilderMixin <|-- MetaMappedArray : metaclass
    MetaPlotsBuilderMixin <|-- PlotsBuilderMixin : metaclass
    Wrapping <|-- MappedArray
    StatsBuilderMixin <|-- MappedArray
    PlotsBuilderMixin <|-- MappedArray
    MetaMappedArray <|-- MappedArray : metaclass
```

# `class MappedArray(Wrapping, StatsBuilderMixin, PlotsBuilderMixin, metaclass=MetaMappedArray)`

## `__init__`

参数：
- `wrapper (ArrayWrapper)`: 数组包装器，核心元数据容器
  - 包含数据行的 `index`（如交易日期）
  - 包含数据列的 `columns`（如股票代码）
  - 包含分组信息 `grouper`（如行业分类）
    
- `mapped_arr (array_like)`: 存储实际的数据值
  - 一维数组，包含所有的数据点
  - 长度必须与 `col_arr` 相同
    
- `col_arr (array_like)`: 列数组，标识数据点的列归属
  - 一维整数数组，值为0, 1, 2, ...
  - 表示对应 `wrapper.columns` 中的哪个列
    
- `id_arr (array_like, optional)`: ID 数组，`mapped_arr` 中每个数据点的唯一标识
  - 如果未提供，自动生成连续的整数 ID
  - 用于跟踪数据点的原始顺序
  - 在数据过滤和排序时保持数据的可追溯性
    
- `idx_arr (array_like, optional)`: 行数组，标识数据点的行归属
  - 一维整数数组，值为 0, 1, 2, ...
  - 表示对应 `wrapper.index` 中的哪个行
    
- `mapping (MappingLike, optional)`: 值映射，数值到标签的转换
  - 字典、命名元组或可调用对象
  - 用于将数值转换为可读的标签
    
- `col_mapper (ColumnMapper, optional)`: 列映射器，优化列操作
  - 如果未提供，会自动创建
  - 用于高效的列数据访问和索引
  - 依赖于 `wrapper` 和 `col_arr`
    
- `**kwargs`: 其他配置参数

```python
def __init__(self,
                wrapper: ArrayWrapper,
                mapped_arr: tp.ArrayLike,
                col_arr: tp.ArrayLike,
                id_arr: tp.Optional[tp.ArrayLike] = None,
                idx_arr: tp.Optional[tp.ArrayLike] = None,
                mapping: tp.Optional[tp.MappingLike] = None,
                col_mapper: tp.Optional[ColumnMapper] = None,
                **kwargs) -> None:
    Wrapping.__init__(
        self,
        wrapper,
        mapped_arr=mapped_arr,
        col_arr=col_arr,
        id_arr=id_arr,
        idx_arr=idx_arr,
        mapping=mapping,
        col_mapper=col_mapper,
        **kwargs
    )
    StatsBuilderMixin.__init__(self)

    mapped_arr = np.asarray(mapped_arr)
    col_arr = np.asarray(col_arr)
    checks.assert_shape_equal(mapped_arr, col_arr, axis=0)
    if id_arr is None:
        id_arr = np.arange(len(mapped_arr))
    else:
        id_arr = np.asarray(id_arr)
    if idx_arr is not None:
        idx_arr = np.asarray(idx_arr)
        checks.assert_shape_equal(mapped_arr, idx_arr, axis=0)
    if mapping is not None:
        if isinstance(mapping, str):
            if mapping.lower() == 'index':
                mapping = self.wrapper.index
            elif mapping.lower() == 'columns':
                mapping = self.wrapper.columns
        mapping = to_mapping(mapping)

    self._mapped_arr = mapped_arr
    self._id_arr = id_arr
    self._col_arr = col_arr
    self._idx_arr = idx_arr
    self._mapping = mapping
    if col_mapper is None:
        col_mapper = ColumnMapper(wrapper, col_arr)
    self._col_mapper = col_mapper
```

## indexing_func_meta
返回执行索引操作 `pd_indexing_func` 后的元数据：
```python
IndexingMetaT = tp.Tuple[
    ArrayWrapper,             # 新包装器
    tp.Array1d,               # 新数据数组
    tp.Array1d,               # 新列索引数组
    tp.Array1d,               # 新ID数组
    tp.Optional[tp.Array1d],  # 新行索引数组
    tp.MaybeArray,            # 选择后的组索引
    tp.Array1d                # 选择后的列索引
]
```

```python
def indexing_func_meta(self, pd_indexing_func: tp.PandasIndexingFunc, **kwargs) -> IndexingMetaT:
    
    new_wrapper, _, group_idxs, col_idxs = \
        self.wrapper.indexing_func_meta(pd_indexing_func, column_only_select=True, **kwargs)
    new_indices, new_col_arr = self.col_mapper._col_idxs_meta(col_idxs)
    new_mapped_arr = self.values[new_indices]
    new_id_arr = self.id_arr[new_indices]
    if self.idx_arr is not None:
        new_idx_arr = self.idx_arr[new_indices]
    else:
        new_idx_arr = None
    return new_wrapper, new_mapped_arr, new_col_arr, new_id_arr, new_idx_arr, group_idxs, col_idxs
```

### 例子

In [16]:
import numpy as np
import vectorbt as vbt
import pandas as pd
from vectorbt.records.mapped_array import MappedArray
from vectorbt.base.array_wrapper import ArrayWrapper

# 创建测试数据
prices = np.array([100, 101, 102, 98, 99, 103])
stocks = np.array([0, 0, 1, 1, 2, 2])
days = np.array([0, 1, 0, 1, 0, 1])

wrapper = ArrayWrapper(
    index=pd.date_range('2023-01-01', periods=2, freq='D'),
    columns=['AAPL', 'GOOGL', 'MSFT'],
    ndim=2
)
ma = MappedArray(wrapper, prices, stocks, idx_arr=days)

# 定义索引函数
def select_apple(x):
    return x['AAPL']  # 选择苹果股票

def select_tech_stocks(x):
    return x[['AAPL', 'GOOGL']]  # 选择科技股

# 执行索引操作并获取元数据
meta = ma.indexing_func_meta(select_tech_stocks)
new_wrapper, new_mapped_arr, new_col_arr, new_id_arr, new_idx_arr, group_idxs, col_idxs = meta

print(f"选择后的数据: {new_mapped_arr}")
print(f"选择后的列: {new_col_arr}")
print(f"选择后的索引: {new_idx_arr}")
print(f"选择后的ID: {new_id_arr}")
print(f"选择后的组索引: {group_idxs}")
print(f"选择后的列索引: {col_idxs}")


选择后的数据: [100 101 102  98]
选择后的列: [0 0 1 1]
选择后的索引: [0 1 0 1]
选择后的ID: [0 1 2 3]
选择后的组索引: [0 1]
选择后的列索引: [0 1]


## indexing_func
执行索引操作  `indexing_func_meta`，然后根据结果构造新的 `MappedArray` 实例返回。

```python
def indexing_func(self: MappedArrayT, pd_indexing_func: tp.PandasIndexingFunc, **kwargs) -> MappedArrayT:

    new_wrapper, new_mapped_arr, new_col_arr, new_id_arr, new_idx_arr, _, _ = \
        self.indexing_func_meta(pd_indexing_func, **kwargs)
    return self.replace(
        wrapper=new_wrapper,
        mapped_arr=new_mapped_arr,
        col_arr=new_col_arr,
        id_arr=new_id_arr,
        idx_arr=new_idx_arr
    )
```

## sort

参数：
- `incl_id` (bool, 可选): 是否将 ID 数组 `self.id_arr` 作为次要排序键
    - False (默认)：只按 `self.col_arr` 排序
    - True：按 `(self.col_arr, self.id_arr)`进行字典序排序
- `idx_arr` (array_like, 可选)：替代的索引数组
    - 如果未提供，使用实例的 `self.idx_arr`
    - 如果提供，会一起进行排序
- `group_by` (GroupByLike, 可选): 分组方式
    - 排序后应用分组
- `**kwargs`：传递给 `replace` 方法的其他参数

返回值：排序后的新 `MappedArray` 实例

```python
def sort(self: MappedArrayT,
            incl_id: bool = False,
            idx_arr: tp.Optional[tp.Array1d] = None,
            group_by: tp.GroupByLike = None,
            **kwargs) -> MappedArrayT:

    if idx_arr is None:
        idx_arr = self.idx_arr
    if self.is_sorted(incl_id=incl_id):
        return self.replace(idx_arr=idx_arr, **kwargs).regroup(group_by)
    if incl_id:
        ind = np.lexsort((self.id_arr, self.col_arr))  # expensive!
    else:
        ind = np.argsort(self.col_arr)
    return self.replace(
        mapped_arr=self.values[ind],
        col_arr=self.col_arr[ind],
        id_arr=self.id_arr[ind],
        idx_arr=idx_arr[ind] if idx_arr is not None else None,
        **kwargs
    ).regroup(group_by)
```

### 例子

In [17]:
import numpy as np
from vectorbt.records.mapped_array import MappedArray
from vectorbt.base.array_wrapper import ArrayWrapper

# 创建未排序的数据
prices = np.array([100, 98, 102, 101, 99, 103])
stocks = np.array([0, 2, 1, 0, 2, 1])  # 未按列排序
ids = np.array([1, 2, 3, 4, 5, 6])
wrapper = ArrayWrapper(index=pd.date_range('2023-01-01', periods=2, freq='D'), columns=['AAPL', 'GOOGL', 'MSFT'], ndim=2)

ma_unsorted = MappedArray(wrapper, prices, stocks, id_arr=ids)
print("===============排序前===============")
print(f"排序前数据: {ma_unsorted.values}")
print(f"排序前列: {ma_unsorted.col_arr}")
print(f"排序前ID: {ma_unsorted.id_arr}")

# 1. 按列排序（不包含ID）
ma_sorted_col = ma_unsorted.sort(incl_id=False)
print("===============不包含ID按列排序后===============")
print(f"排序后数据: {ma_sorted_col.values}")
print(f"排序后列: {ma_sorted_col.col_arr}")
print(f"排序后ID: {ma_sorted_col.id_arr}")

# 2. 按列和ID排序（字典序）
ma_sorted_both = ma_unsorted.sort(incl_id=True)
print("===============包含ID按列排序后===============")
print(f"排序后数据: {ma_sorted_both.values}")
print(f"排序后列: {ma_sorted_both.col_arr}")
print(f"排序后ID: {ma_sorted_both.id_arr}")

===============排序前===============
排序前数据: [100  98 102 101  99 103]
排序前列: [0 2 1 0 2 1]
排序前ID: [1 2 3 4 5 6]
===============不包含ID按列排序后===============
排序后数据: [100 101 102 103  98  99]
排序后列: [0 0 1 1 2 2]
排序后ID: [1 4 3 6 2 5]
===============包含ID按列排序后===============
排序后数据: [100 101 102 103  98  99]
排序后列: [0 0 1 1 2 2]
排序后ID: [1 4 3 6 2 5]


## apply_mask
使用掩码 `mask`（布尔数组）过滤 `MappedArray` 实例。

参数：
- `mask` (array_like): 布尔掩码数组
    - 长度必须与 `self.mapped_arr` 相同
- `idx_arr` (array_like, 可选): 替代的索引数组
    - 如果未提供，使用实例的 `self.idx_arr`
    - 会一起进行过滤
- `group_by` (GroupByLike, 可选): 分组方式
    - 过滤后应用分组
- `**kwargs`: 传递给replace方法的其他参数

返回值：过滤后的新 `MappedArray` 实例

```python
def apply_mask(self: MappedArrayT,
                mask: tp.Array1d,
                idx_arr: tp.Optional[tp.Array1d] = None,
                group_by: tp.GroupByLike = None,
                **kwargs) -> MappedArrayT:

    if idx_arr is None:
        idx_arr = self.idx_arr
    mask_indices = np.flatnonzero(mask)
    return self.replace(
        mapped_arr=np.take(self.values, mask_indices),
        col_arr=np.take(self.col_arr, mask_indices),
        id_arr=np.take(self.id_arr, mask_indices),
        idx_arr=np.take(idx_arr, mask_indices) if idx_arr is not None else None,
        **kwargs
    ).regroup(group_by)
```

### 例子

In [18]:
import numpy as np
from vectorbt.records.mapped_array import MappedArray
from vectorbt.base.array_wrapper import ArrayWrapper

# 创建测试数据
prices = np.array([100, 101, 98, 99, 102, 103, 95, 96])
stocks = np.array([0, 0, 1, 1, 2, 2, 0, 0])
ids = np.array([1, 2, 3, 4, 5, 6, 7, 8])
wrapper = ArrayWrapper(index=pd.date_range('2023-01-01', periods=2, freq='D'), 
                       columns=['AAPL', 'GOOGL', 'MSFT'], 
                       ndim=2)

ma = MappedArray(wrapper, prices, stocks, id_arr=ids)
print(f"原始数据: {ma.values}")
print(f"原始列: {ma.col_arr}")
print(f"原始ID: {ma.id_arr}")

# 1. 基于价格的过滤
high_price_mask = ma.values > 100
print("==================保留价格大于100的过滤===============")
ma_high_price = ma.apply_mask(high_price_mask)
print(f"高价格数据: {ma_high_price.values}")
print(f"高价格列: {ma_high_price.col_arr}")
print(f"高价格ID: {ma_high_price.id_arr}")

# 2. 基于列的过滤
apple_mask = ma.col_arr == 0  # 只保留苹果股票
print("==================保留苹果股票的过滤===============")
ma_apple = ma.apply_mask(apple_mask)
print(f"苹果股票数据: {ma_apple.values}")
print(f"苹果股票列: {ma_apple.col_arr}")
print(f"苹果股票ID: {ma_apple.id_arr}")


原始数据: [100 101  98  99 102 103  95  96]
原始列: [0 0 1 1 2 2 0 0]
原始ID: [1 2 3 4 5 6 7 8]
==================保留价格大于100的过滤===============
高价格数据: [101 102 103]
高价格列: [0 2 2]
高价格ID: [2 5 6]
==================保留苹果股票的过滤===============
苹果股票数据: [100 101  95  96]
苹果股票列: [0 0 0 0]
苹果股票ID: [1 2 7 8]


## map_to_mask
使用 `group_by` 分组后，使用 `inout_map_func_nb` 分别处理每一列对应的 `self.mapped_arr` 中的数据，生成一个布尔数组。

```python
def map_to_mask(self, inout_map_func_nb: tp.MaskInOutMapFunc, *args,
                group_by: tp.GroupByLike = None) -> tp.Array1d:

    # 返回的是 group_by 分组后：
    # (列0,列1,... 分别在 self.mapped_arr 中的对应索引, 各列在self.mapped_arr 中的数量)
    col_map = self.col_mapper.get_col_map(group_by=group_by)
    # 使用 inout_map_func_nb 分别处理每列对应的数据，生成一个与 self.values 长度一致的布尔数组
    return nb.mapped_to_mask_nb(self.values, col_map, inout_map_func_nb, *args)
```

### 例子

In [20]:
import numpy as np
import pandas as pd
from vectorbt.records.mapped_array import MappedArray
from vectorbt.base.array_wrapper import ArrayWrapper
from numba import njit

# 创建测试数据
prices = np.array([100, 101, 98, 99, 102, 103, 95, 96])
stocks = np.array([0, 0, 1, 1, 2, 2, 0, 0])
wrapper = ArrayWrapper(
    index=pd.date_range('2023-01-01', periods=2, freq='D'),
    columns=['AAPL', 'GOOGL', 'MSFT'], 
    ndim=2
)

ma = MappedArray(wrapper, prices, stocks)

@njit
def above_threshold_nb(inout, idxs, col, values, threshold):
    """保留高于阈值的数据点"""
    mask = values > threshold
    inout[idxs[mask]] = True  # 直接修改 inout 参数

# 应用阈值过滤
threshold_mask = ma.map_to_mask(above_threshold_nb, 100)
print(f"原始数据: {ma.values}")
print(f"原始列: {ma.col_arr}")
print(f"阈值掩码: {threshold_mask}")

@njit
def above_mean_nb(inout, idxs, col, values):
    """保留高于列内平均值的数据点"""
    if len(values) == 0:
        return
    mean_val = np.mean(values)
    mask = values > mean_val
    inout[idxs[mask]] = True

# 应用相对过滤
mean_mask = ma.map_to_mask(above_mean_nb)
print(f"高于列内均值的掩码: {mean_mask}")

原始数据: [100 101  98  99 102 103  95  96]
原始列: [0 0 1 1 2 2 0 0]
阈值掩码: [False  True False False  True  True False False]
高于列内均值的掩码: [ True  True False  True False  True False False]


### top_n_mask
选择每列中值最大的前 n 个元素。

```python
@cached_method
def top_n_mask(self, n: int, **kwargs) -> tp.Array1d:
    return self.map_to_mask(nb.top_n_inout_map_nb, n, **kwargs)
```

## apply
在每列上应用函数 `apply_func_nb`。

参数：
- `apply_func_nb` (MappedApplyFunc): Numba编译的应用函数
    - 函数签名：func(idxs, col, values, *args) -> new_values
    - idxs: 当前列/组中数据点的索引
    - col: 当前列/组的标识符
    - values: 当前列/组的数据值
    - *args: 额外的参数
    - 返回值: 变换后的数据值数组
- `*args`: 传递给应用函数的额外参数
- `group_by` (GroupByLike, 可选): 分组方式
    - 在每个组内应用函数
- `apply_per_group` (bool, 可选): 是否按组应用
    - False (默认): 按列应用，忽略分组
    - True: 按组应用，考虑分组
- `dtype` (DTypeLike, 可选): 输出数据类型
    - 如果未指定，保持原数据类型
- `**kwargs`: 传递给replace方法的其他参数

返回值：
    MappedArrayT: 包含变换后数据的新MappedArray实例

```python
def apply(self: MappedArrayT,
            apply_func_nb: tp.MappedApplyFunc, *args,
            group_by: tp.GroupByLike = None,
            apply_per_group: bool = False,
            dtype: tp.Optional[tp.DTypeLike] = None,
            **kwargs) -> MappedArrayT:

    checks.assert_numba_func(apply_func_nb)
    if apply_per_group:
        col_map = self.col_mapper.get_col_map(group_by=group_by)
    else:
        col_map = self.col_mapper.get_col_map(group_by=False)
    mapped_arr = nb.apply_on_mapped_nb(self.values, col_map, apply_func_nb, *args)
    mapped_arr = np.asarray(mapped_arr, dtype=dtype)
    return self.replace(mapped_arr=mapped_arr, **kwargs).regroup(group_by)
```

### 例子

In [ ]:
import numpy as np
import pandas as pd
from vectorbt.records.mapped_array import MappedArray
from vectorbt.base.array_wrapper import ArrayWrapper
from numba import njit

# 创建测试数据
prices = np.array([100, 101, 98, 99, 102, 103])
stocks = np.array([0, 0, 1, 1, 2, 2])
wrapper = ArrayWrapper(
    index=pd.date_range('2023-01-01', periods=2, freq='D'),
    columns=['AAPL', 'GOOGL', 'MSFT'], 
    ndim=2
)

ma = MappedArray(wrapper, prices, stocks)
print(f"原始数据: {ma.values}")

@njit
def calculate_returns_nb(idxs, col, values):
    """计算每列的收益率"""
    if len(values) < 2:
        return np.full_like(values, np.nan, dtype=np.float64)
    
    returns = np.empty_like(values, dtype=np.float64)  # 显式指定类型
    returns[0] = np.nan
    for i in range(1, len(values)):
        returns[i] = (values[i] - values[i-1]) / values[i-1]
    return returns

returns_ma = ma.apply(calculate_returns_nb)
print(f"收益率: {returns_ma.values}")

@njit
def moving_average_nb(idxs, col, values, window):
    """计算移动平均"""
    if len(values) < window:
        return np.full_like(values, np.nan, dtype=np.float64)
    
    ma_values = np.full_like(values, np.nan, dtype=np.float64)
    for i in range(window-1, len(values)):
        ma_values[i] = np.mean(values[i-window+1:i+1])
    
    return ma_values

ma_2period = ma.apply(moving_average_nb, 2)
print(f"2期移动平均: {ma_2period.values}")

原始数据: [100 101  98  99 102 103]
收益率: [       nan 0.01              nan 0.01020408        nan 0.00980392]
2期移动平均: [  nan 100.5   nan  98.5   nan 102.5]


## reduce
聚合：使用自定义的 Numba 函数将每个列/组的多个数据点归约为单个值或数组。

参数：
- `reduce_func_nb` (ReduceFunc): Numba编译的归约函数
    - 函数签名：func(idxs, col, values, *args) -> result
    - idxs: 当前列/组中数据点的索引
    - col: 当前列/组的标识符
    - values: 当前列/组的数据值
    - *args: 额外的参数
    - 返回值: 归约后的结果（标量或数组）
- `*args`: 传递给归约函数的额外参数
- `idx_arr` (array_like, 可选): 索引数组
    - 如果returns_idx=True，必须提供
    - 用于返回索引位置而非值
- `returns_array` (bool, 可选): 是否返回数组结果
    - False (默认): 返回标量结果
    - True: 返回数组结果
- `returns_idx` (bool, 可选): 是否返回索引
    - False (默认): 返回数据值
    - True: 返回索引位置
- `to_index` (bool, 可选): 是否转换为索引标签
    - True (默认): 返回索引标签
    - False: 返回原始位置
- `fill_value` (Scalar, 可选): 空值填充
    - 默认: np.nan
    - 用于填充空列/组的结果
- `group_by` (GroupByLike, 可选): 分组方式
    - 在每个组内进行归约
- `wrap_kwargs` (dict, 可选): 包装参数
    - 传递给wrapper.wrap的额外参数

返回值：`tp.MaybeSeriesFrame: pandas Series`或 `DataFrame`，取决于返回类型

```python
def reduce(self,
            reduce_func_nb: tp.ReduceFunc, *args,
            idx_arr: tp.Optional[tp.Array1d] = None,
            returns_array: bool = False,
            returns_idx: bool = False,
            to_index: bool = True,
            fill_value: tp.Scalar = np.nan,
            group_by: tp.GroupByLike = None,
            wrap_kwargs: tp.KwargsLike = None) -> tp.MaybeSeriesFrame:

    checks.assert_numba_func(reduce_func_nb)
    if idx_arr is None:
        if self.idx_arr is None:
            if returns_idx:
                raise ValueError("Must pass idx_arr")
        idx_arr = self.idx_arr

    col_map = self.col_mapper.get_col_map(group_by=group_by)
    if not returns_array:
        if not returns_idx:
            out = nb.reduce_mapped_nb(
                self.values,
                col_map,
                fill_value,
                reduce_func_nb,
                *args
            )
        else:
            out = nb.reduce_mapped_to_idx_nb(
                self.values,
                col_map,
                idx_arr,
                fill_value,
                reduce_func_nb,
                *args
            )
    else:
        if not returns_idx:
            out = nb.reduce_mapped_to_array_nb(
                self.values,
                col_map,
                fill_value,
                reduce_func_nb,
                *args
            )
        else:
            out = nb.reduce_mapped_to_idx_array_nb(
                self.values,
                col_map,
                idx_arr,
                fill_value,
                reduce_func_nb,
                *args
            )

    wrap_kwargs = merge_dicts(dict(
        name_or_index='reduce' if not returns_array else None,
        to_index=returns_idx and to_index,
        fillna=-1 if returns_idx else None,
        dtype=np.int64 if returns_idx else None
    ), wrap_kwargs)
    return self.wrapper.wrap_reduced(out, group_by=group_by, **wrap_kwargs)
```

### 例子

In [33]:
import numpy as np
from vectorbt.records.mapped_array import MappedArray
from vectorbt.base.array_wrapper import ArrayWrapper
from numba import njit

# 创建测试数据
values = np.array([1, 5, 2, 8, 3, 9, 4, 7, 6])
stocks = np.array([0, 0, 0, 1, 1, 1, 2, 2, 2])
days = np.array([0, 1, 2, 0, 1, 2, 0, 1, 2])

wrapper = ArrayWrapper(
    index=pd.date_range('2023-01-01', periods=3, freq='D'),
    columns=['AAPL', 'GOOGL', 'MSFT'],
    ndim=2
)

ma = MappedArray(wrapper, values, stocks, idx_arr=days)
print(ma.to_pd())

            AAPL  GOOGL  MSFT
2023-01-01   1.0    8.0   4.0
2023-01-02   5.0    3.0   7.0
2023-01-03   2.0    9.0   6.0


#### 计算均值

In [30]:
# 1. 计算均值（标量结果）
@njit
def mean_nb(col, values):
    """计算平均值"""
    if len(values) == 0:
        return np.nan
    return np.mean(values)

# 计算均值
mean_result = ma.reduce(mean_nb)
print(f"均值结果: {mean_result}")

均值结果: AAPL     2.666667
GOOGL    6.666667
MSFT     5.666667
Name: reduce, dtype: float64


#### 找到最大值的位置

In [31]:
# 2. 找到最大值的位置（索引结果）
@njit
def argmax_nb(col, values):
    """找到最大值的索引"""
    if len(values) == 0:
        return -1
    return np.argmax(values)

# 找到最大值位置
max_positions = ma.reduce(argmax_nb, returns_idx=True)
print(f"最大值位置: {max_positions}")

最大值位置: AAPL    2023-01-02
GOOGL   2023-01-03
MSFT    2023-01-02
Name: reduce, dtype: datetime64[ns]


#### 计算分位数

In [32]:
# 3. 计算分位数（数组结果）
@njit
def quantiles_nb(col, values, q_values):
    """计算分位数"""
    if len(values) == 0:
        return np.full(len(q_values), np.nan)
    
    sorted_values = np.sort(values)
    n = len(sorted_values)
    quantiles = np.empty(len(q_values))
    
    for i, q in enumerate(q_values):
        pos = q * (n - 1)
        lower = int(pos)
        upper = min(lower + 1, n - 1)
        
        if lower == upper:
            quantiles[i] = sorted_values[lower]
        else:
            weight = pos - lower
            quantiles[i] = sorted_values[lower] * (1 - weight) + sorted_values[upper] * weight
    
    return quantiles

# 计算分位数
q_values = np.array([0.25, 0.5, 0.75])
quantile_result = ma.reduce(quantiles_nb, q_values, returns_array=True)
print(f"分位数结果:\n{quantile_result}")

分位数结果:
   AAPL  GOOGL  MSFT
0   1.5    5.5   5.0
1   2.0    8.0   6.0
2   3.5    8.5   6.5


## describe

```python
@cached_method
def describe(self,
                percentiles: tp.Optional[tp.ArrayLike] = None,
                ddof: int = 1,
                group_by: tp.GroupByLike = None,
                wrap_kwargs: tp.KwargsLike = None,
                **kwargs) -> tp.SeriesFrame:
    """Return statistics by column/group."""
    if percentiles is not None:
        percentiles = to_1d_array(percentiles)
    else:
        percentiles = np.array([0.25, 0.5, 0.75])
    percentiles = percentiles.tolist()
    if 0.5 not in percentiles:
        percentiles.append(0.5)
    percentiles = np.unique(percentiles)
    perc_formatted = pd.io.formats.format.format_percentiles(percentiles)
    index = pd.Index(['count', 'mean', 'std', 'min', *perc_formatted, 'max'])
    wrap_kwargs = merge_dicts(dict(name_or_index=index), wrap_kwargs)
    out = self.reduce(
        generic_nb.describe_reduce_nb,
        percentiles,
        ddof,
        returns_array=True,
        returns_idx=False,
        group_by=group_by,
        wrap_kwargs=wrap_kwargs,
        **kwargs
    )
    if isinstance(out, pd.DataFrame):
        out.loc['count'].fillna(0., inplace=True)
    else:
        if np.isnan(out.loc['count']):
            out.loc['count'] = 0.
    return out
```

### 例子